# Day 7 — Delta, Vega, Gamma and Discrete-Trigger Smoothing

## tl;dr

- **Status: PASS with documented Gamma caveats.** All 640 requested full-product
  Greek replication rows are present, every base/±h group uses CRN/identical
  scrambles, and all nine Day 7 delivery gates pass.
- At the 0.5% active-asset bump in **near-KI**, M3 reduces replication SD versus
  equal-outer-(N) M0 for Delta (0.1302→0.0993), Vega (0.0885→0.0473) and Gamma
  (0.2146→0.1045).
- In **near-autocall**, Brownian-bridge KI smoothing does not remove the discrete
  trigger: M3 active Delta/Gamma SD is higher than M0 (0.3540 vs 0.1211 and
  1.5508 vs 0.5798), while Vega SD is lower (0.0294 vs 0.0526).
- Pure bump-shape Gamma plateau coverage is only **12.5%**. All Gamma estimates
  remain uncertainty-consistent at the four-replication budget, but that is not
  presented as a true plateau and no unstable bump is hidden.
- On the isolated next-autocall redemption component, AC-Smooth agrees in value
  with raw RQMC (paired difference 0.00558/100) and reduces price replication SD
  by **97.0×**; active Delta and Gamma SD fall by about **61×** and **126×**.
- Changing the fresh nested bridge-bank seed has negligible effect on the tested
  near-KI active Delta/Gamma relative to outer-replication uncertainty.

## Context & Methods

Day 6 passed the full price gate, so Day 7 evaluates whether the price estimator
improvement transfers to finite-difference Greeks. The full product comparison is:

- **M0:** direct fine-grid Monte Carlo;
- **M3:** observation-endpoint scrambled-Sobol RQMC with three-asset
  Brownian-bridge continuous-KI conditioning.

For each estimator, scenario and replication, base/(+h)/(-h) valuations reuse
the same random seed or Sobol scramble. Contractual reference levels, absolute KI
barriers and discrete trigger ratios remain fixed when spot is bumped.

### Key assumptions

- **near-KI:** RC-A starts from `(0.765, 1.000, 1.000)`, conditional on monitoring
  beginning at that state; asset 1 is the active/worst asset and lies 1.5 points
  above the 0.75 continuous KI barrier.
- **near-autocall:** RC-A is rolled to two days before its first remaining autocall
  date and starts from `(1.004, 1.001, 0.998)`, conditional on no prior KI and no
  accrued coupon memory; asset 3 is the active/worst asset.
- Spot relative bumps are `0.1%, 0.25%, 0.5%, 1.0%`; volatility bumps are
  `0.25, 0.50, 1.00` volatility points.
- The full-product experiment is a diagnostic budget (not the final Day 9 main
  experiment): M0 uses 512 paths and 252 steps/year; M3 uses 512 outer paths and
  a 128×12 fresh nested bridge bank, each with four independent replications.

### B8 scope

`AC-Smooth` is deliberately limited to the next-observation autocall **principal
redemption component**. A Householder rotation makes the final independent normal
load positively into all three assets, so the worst-of call becomes a scalar
threshold and its conditional probability replaces the indicator. This local
diagnostic is not labelled as Brownian-bridge conditioning and does not replace
the full-product near-autocall experiment, which retains future-cashflow effects.

### Chart contract

1. **Bump stability:** three separate two-panel figures (Delta, Vega and Gamma),
   with near-KI and near-autocall shown side by side. Each panel keeps the full
   bump grid and 95% replication intervals.
2. **Plateau coverage:** one annotated heatmap of pass rates across scenario,
   Greek and method.
3. **AC-Smooth diagnostic:** one dimensionless horizontal bar chart of the
   raw-RQMC / Smooth-RQMC replication-SD reduction factor for price, Delta and
   Gamma; the price-agreement gap is stated in the subtitle.
4. **Estimator-regime stability summary:** one 2 x 3 diverging matrix of the
   active-asset `M0 SD / M3 SD` ratio at the common middle bump. Values above
   one favour M3; values below one favour M0.

Every exported figure contains at most two panels. All figures share the same
font scale, title hierarchy, method colours, light grid and 200 dpi export.

In [ ]:
from pathlib import Path
import json
import math
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "core_project_config.json").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config" / "core_project_config.json").is_file():
    raise FileNotFoundError("Start Jupyter from inside the autocallable-rqmc repository")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from autocallable_bb import simulate_conditioned_replication
from autocallable_direct import make_psd_correlation, parse_research_contract, simulate_direct_replication, stable_seed
from autocallable_greeks import (
    autocall_redemption_replication,
    central_first_derivative,
    central_second_derivative,
    roll_contract_to_autocall,
    stable_autocall_rotation,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 190)
pd.options.display.float_format = "{:,.8f}".format
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "axes.labelsize": 10,
    "axes.edgecolor": "#475569",
    "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
    "grid.color": "#dbe3ec",
    "grid.linewidth": 0.75,
    "grid.alpha": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

CONFIG_PATH = PROJECT_ROOT / "config" / "core_project_config.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "day7_greeks_trigger_smoothing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = json.load(handle)

SEED = 20260807
SPOT_BUMP_FRACTIONS = np.array([0.0010, 0.0025, 0.0050, 0.0100])
VOL_BUMPS = np.array([0.0025, 0.0050, 0.0100])
REPLICATIONS = 4
M0_PATHS = 512
M0_STEPS_PER_YEAR = 252
M3_PATHS = 512
M3_INNER_PATHS = 128
M3_BRIDGE_SUBSTEPS = 12
AC_PATHS = 4096
AC_REPLICATIONS = 16
print("Project:", PROJECT_ROOT)
print("Evidence:", OUTPUT_DIR)

## Data

In [ ]:
def is_number(value):
    return isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool) and np.isfinite(value)


def read_market_inputs(path):
    workbook = openpyxl.load_workbook(path, data_only=True, read_only=False)
    setup = workbook["Setup"]
    snapshot = workbook["Underlying_Snapshot"]
    history = workbook["Underlying_History"]
    market_history = workbook["Market_History"]
    tickers = [snapshot.cell(row, 2).value for row in (6, 7, 8)]
    dividend_yields = np.array([snapshot.cell(row, 5).value for row in (6, 7, 8)], dtype=float) / 100.0
    volatilities = np.array([snapshot.cell(row, 8).value for row in (6, 7, 8)], dtype=float) / 100.0
    histories = []
    for date_column, value_column, ticker in zip((1, 4, 7), (2, 5, 8), tickers):
        values = {}
        for row in range(6, history.max_row + 1):
            date_value = history.cell(row, date_column).value
            level = history.cell(row, value_column).value
            if hasattr(date_value, "year") and is_number(level) and level > 0:
                values[pd.Timestamp(date_value)] = float(level)
        histories.append(pd.Series(values, name=ticker).sort_index())
    prices = pd.concat(histories, axis=1, join="inner").dropna()
    correlation = make_psd_correlation(np.log(prices / prices.shift(1)).dropna().corr().to_numpy())
    rates = []
    for row in range(6, market_history.max_row + 1):
        date_value = market_history.cell(row, 10).value
        rate_pct = market_history.cell(row, 11).value
        if hasattr(date_value, "year") and is_number(rate_pct) and rate_pct > 0:
            rates.append((pd.Timestamp(date_value), float(rate_pct) / 100.0))
    rates = pd.Series(dict(rates)).sort_index()
    return {
        "tickers": tickers,
        "dividend_yields": dividend_yields,
        "volatilities": volatilities,
        "correlation": correlation,
        "risk_free_rate": float(rates.iloc[-1]),
        "rate_as_of": rates.index[-1],
        "workbook_as_of": pd.Timestamp(setup["B8"].value),
    }


market = read_market_inputs(PROJECT_ROOT / CONFIG["market_data"]["relative_path"])
rc_a = parse_research_contract(CONFIG, "RC-A")
near_autocall_contract = roll_contract_to_autocall(rc_a, source_observation_index=1, first_time_years=2.0 / 365.0)
scenarios = {
    "near-KI": {
        "contract": rc_a,
        "initial_spot_ratios": np.array([0.765, 1.000, 1.000]),
        "active_asset": 0,
        "description": "RC-A at issue-style horizon; asset 1 just above continuous KI",
    },
    "near-autocall": {
        "contract": near_autocall_contract,
        "initial_spot_ratios": np.array([1.004, 1.001, 0.998]),
        "active_asset": 2,
        "description": "RC-A remaining life; two days to first remaining autocall",
    },
}
scenario_definitions = pd.DataFrame([
    {
        "scenario": name,
        "contract_id": values["contract"].contract_id,
        "maturity_years": values["contract"].maturity_years,
        "first_observation_years": values["contract"].observation_times[0],
        "initial_spot_1": values["initial_spot_ratios"][0],
        "initial_spot_2": values["initial_spot_ratios"][1],
        "initial_spot_3": values["initial_spot_ratios"][2],
        "active_asset": values["active_asset"] + 1,
        "ki_barrier": values["contract"].ki_barrier_ratio,
        "description": values["description"],
        "claim_boundary": values["contract"].claim_boundary,
    }
    for name, values in scenarios.items()
])
display(scenario_definitions)
print(f"Workbook as of {market['workbook_as_of'].date()}; rate {market['risk_free_rate']:.4%} as of {market['rate_as_of'].date()}")

## Results

### 1. Full-product CRN Delta, Vega and Gamma grid

In [ ]:
def price_full_product(method, scenario_name, spots, volatilities, replication, bridge_bank_seed, bank_cache):
    scenario = scenarios[scenario_name]
    contract = scenario["contract"]
    pricing_seed = stable_seed(SEED, scenario_name, method, "outer", replication)
    if method == "M0":
        return simulate_direct_replication(
            contract=contract,
            risk_free_rate=market["risk_free_rate"],
            dividend_yields=market["dividend_yields"],
            volatilities=volatilities,
            correlation=market["correlation"],
            annual_coupon=contract.baseline_annual_coupon,
            n_paths=M0_PATHS,
            steps_per_year=M0_STEPS_PER_YEAR,
            seed=pricing_seed,
            batch_size=512,
            initial_spot_ratios=spots,
        ), pricing_seed
    if method == "M3":
        return simulate_conditioned_replication(
            contract=contract,
            risk_free_rate=market["risk_free_rate"],
            dividend_yields=market["dividend_yields"],
            volatilities=volatilities,
            correlation=market["correlation"],
            annual_coupon=contract.baseline_annual_coupon,
            n_paths=M3_PATHS,
            outer_method="rqmc",
            seed=pricing_seed,
            bridge_bank_seed=bridge_bank_seed,
            inner_paths=M3_INNER_PATHS,
            bridge_substeps=M3_BRIDGE_SUBSTEPS,
            bank_cache=bank_cache,
            paired_bernoulli=False,
            initial_spot_ratios=spots,
        ), pricing_seed
    raise KeyError(method)


raw_valuations = []
greek_rows = []
experiment_started = time.perf_counter()
for scenario_name, scenario in scenarios.items():
    base_spots = scenario["initial_spot_ratios"]
    base_sigma = market["volatilities"]
    for method in ("M0", "M3"):
        bridge_bank_seed = stable_seed(SEED, scenario_name, method, "bridge-bank")
        bank_cache = {}
        for replication in range(REPLICATIONS):
            base_result, pricing_seed = price_full_product(
                method, scenario_name, base_spots, base_sigma, replication, bridge_bank_seed, bank_cache
            )
            base_value = base_result["total_value"]
            raw_valuations.append({
                "scenario": scenario_name, "method": method, "replication": replication,
                "greek": "Base", "asset": "base", "bump": 0.0, "direction": "base",
                "total_value": base_value, "runtime_seconds": base_result["runtime_seconds"],
                "pricing_seed": pricing_seed, "bridge_bank_seed": bridge_bank_seed,
            })

            for asset_index in range(3):
                for bump_fraction in SPOT_BUMP_FRACTIONS:
                    bump_amount = base_spots[asset_index] * bump_fraction
                    values = {}
                    runtimes = 0.0
                    for direction, sign in (("minus", -1.0), ("plus", 1.0)):
                        bumped_spots = base_spots.copy()
                        bumped_spots[asset_index] += sign * bump_amount
                        result, pricing_seed = price_full_product(
                            method, scenario_name, bumped_spots, base_sigma, replication,
                            bridge_bank_seed, bank_cache,
                        )
                        values[direction] = result["total_value"]
                        runtimes += result["runtime_seconds"]
                        raw_valuations.append({
                            "scenario": scenario_name, "method": method, "replication": replication,
                            "greek": "Spot", "asset": f"asset_{asset_index + 1}",
                            "bump": bump_fraction, "direction": direction,
                            "total_value": result["total_value"], "runtime_seconds": result["runtime_seconds"],
                            "pricing_seed": pricing_seed, "bridge_bank_seed": bridge_bank_seed,
                        })
                    delta_raw = central_first_derivative(values["plus"], values["minus"], bump_amount)
                    gamma_raw = central_second_derivative(base_value, values["plus"], values["minus"], bump_amount)
                    common = {
                        "scenario": scenario_name, "method": method, "replication": replication,
                        "asset": f"asset_{asset_index + 1}", "asset_index": asset_index + 1,
                        "bump": bump_fraction, "bump_label": f"{100*bump_fraction:.2f}% spot",
                        "active_asset": asset_index == scenario["active_asset"], "runtime_seconds": runtimes,
                        "pricing_seed": pricing_seed, "bridge_bank_seed": bridge_bank_seed,
                    }
                    greek_rows.append({
                        **common, "greek": "Delta", "estimate_raw": delta_raw,
                        "estimate_reported": delta_raw * base_spots[asset_index] * 0.01,
                        "reported_unit": "PV per 1% relative spot move",
                    })
                    greek_rows.append({
                        **common, "greek": "Gamma", "estimate_raw": gamma_raw,
                        "estimate_reported": gamma_raw * base_spots[asset_index] ** 2 * 1e-4,
                        "reported_unit": "PV per squared 1% relative spot move",
                    })

            for bump_fraction in SPOT_BUMP_FRACTIONS:
                values = {}
                runtimes = 0.0
                for direction, sign in (("minus", -1.0), ("plus", 1.0)):
                    bumped_spots = base_spots * (1.0 + sign * bump_fraction)
                    result, pricing_seed = price_full_product(
                        method, scenario_name, bumped_spots, base_sigma, replication,
                        bridge_bank_seed, bank_cache,
                    )
                    values[direction] = result["total_value"]
                    runtimes += result["runtime_seconds"]
                    raw_valuations.append({
                        "scenario": scenario_name, "method": method, "replication": replication,
                        "greek": "ParallelSpot", "asset": "parallel", "bump": bump_fraction,
                        "direction": direction, "total_value": result["total_value"],
                        "runtime_seconds": result["runtime_seconds"], "pricing_seed": pricing_seed,
                        "bridge_bank_seed": bridge_bank_seed,
                    })
                parallel_gamma = central_second_derivative(base_value, values["plus"], values["minus"], bump_fraction)
                greek_rows.append({
                    "scenario": scenario_name, "method": method, "replication": replication,
                    "asset": "parallel", "asset_index": 0, "bump": bump_fraction,
                    "bump_label": f"{100*bump_fraction:.2f}% parallel spot", "active_asset": False,
                    "runtime_seconds": runtimes, "pricing_seed": pricing_seed,
                    "bridge_bank_seed": bridge_bank_seed, "greek": "Parallel Gamma",
                    "estimate_raw": parallel_gamma, "estimate_reported": parallel_gamma * 1e-4,
                    "reported_unit": "PV per squared 1% parallel relative spot move",
                })

            for asset_index in range(3):
                for vol_bump in VOL_BUMPS:
                    values = {}
                    runtimes = 0.0
                    for direction, sign in (("minus", -1.0), ("plus", 1.0)):
                        bumped_sigma = base_sigma.copy()
                        bumped_sigma[asset_index] += sign * vol_bump
                        result, pricing_seed = price_full_product(
                            method, scenario_name, base_spots, bumped_sigma, replication,
                            bridge_bank_seed, bank_cache,
                        )
                        values[direction] = result["total_value"]
                        runtimes += result["runtime_seconds"]
                        raw_valuations.append({
                            "scenario": scenario_name, "method": method, "replication": replication,
                            "greek": "Volatility", "asset": f"asset_{asset_index + 1}",
                            "bump": vol_bump, "direction": direction,
                            "total_value": result["total_value"], "runtime_seconds": result["runtime_seconds"],
                            "pricing_seed": pricing_seed, "bridge_bank_seed": bridge_bank_seed,
                        })
                    vega_raw = central_first_derivative(values["plus"], values["minus"], vol_bump)
                    greek_rows.append({
                        "scenario": scenario_name, "method": method, "replication": replication,
                        "asset": f"asset_{asset_index + 1}", "asset_index": asset_index + 1,
                        "bump": vol_bump, "bump_label": f"{100*vol_bump:.2f} vol points",
                        "active_asset": asset_index == scenario["active_asset"], "runtime_seconds": runtimes,
                        "pricing_seed": pricing_seed, "bridge_bank_seed": bridge_bank_seed,
                        "greek": "Vega", "estimate_raw": vega_raw,
                        "estimate_reported": vega_raw * 0.01, "reported_unit": "PV per 1 volatility point",
                    })

            for vol_bump in VOL_BUMPS:
                values = {}
                runtimes = 0.0
                for direction, sign in (("minus", -1.0), ("plus", 1.0)):
                    bumped_sigma = base_sigma + sign * vol_bump
                    result, pricing_seed = price_full_product(
                        method, scenario_name, base_spots, bumped_sigma, replication,
                        bridge_bank_seed, bank_cache,
                    )
                    values[direction] = result["total_value"]
                    runtimes += result["runtime_seconds"]
                    raw_valuations.append({
                        "scenario": scenario_name, "method": method, "replication": replication,
                        "greek": "ParallelVolatility", "asset": "parallel", "bump": vol_bump,
                        "direction": direction, "total_value": result["total_value"],
                        "runtime_seconds": result["runtime_seconds"], "pricing_seed": pricing_seed,
                        "bridge_bank_seed": bridge_bank_seed,
                    })
                parallel_vega = central_first_derivative(values["plus"], values["minus"], vol_bump)
                greek_rows.append({
                    "scenario": scenario_name, "method": method, "replication": replication,
                    "asset": "parallel", "asset_index": 0, "bump": vol_bump,
                    "bump_label": f"{100*vol_bump:.2f} parallel vol points", "active_asset": False,
                    "runtime_seconds": runtimes, "pricing_seed": pricing_seed,
                    "bridge_bank_seed": bridge_bank_seed, "greek": "Parallel Vega",
                    "estimate_raw": parallel_vega, "estimate_reported": parallel_vega * 0.01,
                    "reported_unit": "PV per 1 parallel volatility point",
                })

greek_replications = pd.DataFrame(greek_rows)
raw_valuations = pd.DataFrame(raw_valuations)
print(f"Full-product Greek grid runtime: {time.perf_counter() - experiment_started:.2f} seconds")
print("Greek replication rows:", len(greek_replications), "raw valuations:", len(raw_valuations))

### 2. Replication uncertainty and bump-size plateau

In [ ]:
greek_summary = (
    greek_replications.groupby(
        ["scenario", "method", "greek", "asset", "asset_index", "active_asset", "bump", "bump_label", "reported_unit"],
        as_index=False,
    )
    .agg(
        replications=("estimate_reported", "count"),
        estimate_mean=("estimate_reported", "mean"),
        estimate_sd=("estimate_reported", "std"),
        runtime_seconds_mean=("runtime_seconds", "mean"),
    )
)
greek_summary["estimate_se"] = greek_summary["estimate_sd"] / np.sqrt(greek_summary["replications"])

plateau_rows = []
for keys, group in greek_summary.groupby(["scenario", "method", "greek", "asset"], sort=False):
    group = group.sort_values("bump")
    # Exclude the deliberately most noise-sensitive 0.1% spot point from the
    # plateau decision while retaining it in every raw/summary output.
    tested = group.iloc[1:] if len(group) == 4 else group
    values = tested["estimate_mean"].to_numpy(float)
    dispersion = float(values.max() - values.min())
    median_abs = float(abs(np.median(values)))
    max_se = float(tested["estimate_se"].max())
    relative_tolerance = 0.50 if keys[2] in {"Gamma", "Parallel Gamma"} else 0.20
    shape_tolerance = relative_tolerance * max(median_abs, 1e-6)
    uncertainty_tolerance = max(3.0 * max_se, shape_tolerance)
    plateau_rows.append({
        "scenario": keys[0], "method": keys[1], "greek": keys[2], "asset": keys[3],
        "tested_bump_min": tested["bump"].min(), "tested_bump_max": tested["bump"].max(),
        "bump_dispersion": dispersion, "median_abs_estimate": median_abs,
        "max_replication_se": max_se,
        "relative_dispersion": dispersion / max(median_abs, 1e-6),
        "shape_plateau_tolerance": shape_tolerance,
        "uncertainty_tolerance": uncertainty_tolerance,
        "plateau_pass": dispersion <= shape_tolerance,
        "uncertainty_consistent": dispersion <= uncertainty_tolerance,
    })
bump_plateau = pd.DataFrame(plateau_rows)
plateau_rate = (
    bump_plateau.groupby(["scenario", "method", "greek"], as_index=False)
    .agg(
        plateau_rate=("plateau_pass", "mean"),
        uncertainty_consistent_rate=("uncertainty_consistent", "mean"),
        groups=("asset", "count"),
    )
)
active_summary = greek_summary[greek_summary["active_asset"]].copy()
display(active_summary[[
    "scenario", "method", "greek", "asset", "bump_label", "estimate_mean", "estimate_se", "reported_unit"
]])
display(plateau_rate)

### 3. Bridge-bank seed sensitivity for the near-KI active Delta/Gamma

In [ ]:
seed_sensitivity_rows = []
scenario_name = "near-KI"
scenario = scenarios[scenario_name]
base_spots = scenario["initial_spot_ratios"]
asset_index = scenario["active_asset"]
bump_fraction = 0.005
bump_amount = base_spots[asset_index] * bump_fraction
for bank_variant in range(3):
    bridge_seed = stable_seed(SEED, "seed-sensitivity", bank_variant)
    bank_cache = {}
    for replication in range(3):
        values = {}
        for direction, sign in (("minus", -1.0), ("base", 0.0), ("plus", 1.0)):
            spots = base_spots.copy()
            spots[asset_index] += sign * bump_amount
            result, pricing_seed = price_full_product(
                "M3", scenario_name, spots, market["volatilities"], replication,
                bridge_seed, bank_cache,
            )
            values[direction] = result["total_value"]
        seed_sensitivity_rows.extend([
            {
                "bank_variant": bank_variant, "bridge_bank_seed": bridge_seed,
                "replication": replication, "greek": "Delta",
                "estimate_reported": central_first_derivative(values["plus"], values["minus"], bump_amount)
                * base_spots[asset_index] * 0.01,
            },
            {
                "bank_variant": bank_variant, "bridge_bank_seed": bridge_seed,
                "replication": replication, "greek": "Gamma",
                "estimate_reported": central_second_derivative(values["base"], values["plus"], values["minus"], bump_amount)
                * base_spots[asset_index] ** 2 * 1e-4,
            },
        ])
bridge_seed_sensitivity = pd.DataFrame(seed_sensitivity_rows)
bridge_seed_sensitivity_summary = (
    bridge_seed_sensitivity.groupby(["bank_variant", "bridge_bank_seed", "greek"], as_index=False)
    .agg(estimate_mean=("estimate_reported", "mean"), estimate_sd=("estimate_reported", "std"))
)
bridge_seed_dispersion = (
    bridge_seed_sensitivity_summary.groupby("greek", as_index=False)
    .agg(
        bank_seed_mean=("estimate_mean", "mean"),
        bank_seed_sd=("estimate_mean", "std"),
        bank_seed_range=("estimate_mean", lambda values: values.max() - values.min()),
    )
)
display(bridge_seed_sensitivity_summary)
display(bridge_seed_dispersion)

### 4. B8 local AC-Smooth diagnostic

In [ ]:
ac_scenario = scenarios["near-autocall"]
ac_contract = ac_scenario["contract"]
ac_initial = ac_scenario["initial_spot_ratios"]
ac_time = ac_contract.observation_times[0]
ac_trigger = float(ac_contract.autocall_trigger_ratios[0])
ac_rows = []
ac_greek_rows = []
ac_methods = (
    ("AC-raw-MC", "mc", False),
    ("AC-raw-RQMC", "rqmc", False),
    ("AC-Smooth-RQMC", "rqmc", True),
)
for estimator, method, smooth in ac_methods:
    for replication in range(AC_REPLICATIONS):
        seed = stable_seed(SEED, "AC-smooth", replication)
        base = autocall_redemption_replication(
            ac_initial, ac_trigger, ac_time, ac_contract.principal,
            market["risk_free_rate"], market["dividend_yields"], market["volatilities"],
            market["correlation"], AC_PATHS, method, seed, smooth,
        )
        ac_rows.append({"estimator": estimator, "replication": replication, "bump": 0.0, "direction": "base", **base})
        for asset_index in range(3):
            for bump_fraction in SPOT_BUMP_FRACTIONS:
                bump_amount = ac_initial[asset_index] * bump_fraction
                values = {}
                diagnostics = []
                for direction, sign in (("minus", -1.0), ("plus", 1.0)):
                    spots = ac_initial.copy()
                    spots[asset_index] += sign * bump_amount
                    row = autocall_redemption_replication(
                        spots, ac_trigger, ac_time, ac_contract.principal,
                        market["risk_free_rate"], market["dividend_yields"], market["volatilities"],
                        market["correlation"], AC_PATHS, method, seed, smooth,
                    )
                    values[direction] = row["value"]
                    diagnostics.append(row)
                    ac_rows.append({
                        "estimator": estimator, "replication": replication, "asset": f"asset_{asset_index + 1}",
                        "bump": bump_fraction, "direction": direction, **row,
                    })
                delta = central_first_derivative(values["plus"], values["minus"], bump_amount)
                gamma = central_second_derivative(base["value"], values["plus"], values["minus"], bump_amount)
                common = {
                    "estimator": estimator, "replication": replication, "asset": f"asset_{asset_index + 1}",
                    "asset_index": asset_index + 1, "active_asset": asset_index == ac_scenario["active_asset"],
                    "bump": bump_fraction, "bump_label": f"{100*bump_fraction:.2f}% spot",
                }
                ac_greek_rows.extend([
                    {**common, "greek": "Delta", "estimate_reported": delta * ac_initial[asset_index] * 0.01},
                    {**common, "greek": "Gamma", "estimate_reported": gamma * ac_initial[asset_index] ** 2 * 1e-4},
                ])

ac_smooth_valuations = pd.DataFrame(ac_rows)
ac_smooth_greeks = pd.DataFrame(ac_greek_rows)
ac_price_summary = (
    ac_smooth_valuations[ac_smooth_valuations["direction"] == "base"]
    .groupby("estimator", as_index=False)
    .agg(
        value_mean=("value", "mean"), value_sd=("value", "std"),
        call_probability_mean=("call_probability", "mean"),
        runtime_seconds_mean=("runtime_seconds", "mean"),
        clipping_rate_max=("probability_clipping_rate", "max"),
        orthogonality_error_max=("orthogonality_error", "max"),
        correlation_error_max=("correlation_error", "max"),
        minimum_final_loading=("minimum_final_loading", "min"),
    )
)
ac_greek_summary = (
    ac_smooth_greeks.groupby(["estimator", "greek", "asset", "asset_index", "active_asset", "bump", "bump_label"], as_index=False)
    .agg(estimate_mean=("estimate_reported", "mean"), estimate_sd=("estimate_reported", "std"))
)
ac_price_pivot = ac_smooth_valuations[ac_smooth_valuations["direction"] == "base"].pivot(
    index="replication", columns="estimator", values="value"
)
ac_price_difference = ac_price_pivot["AC-Smooth-RQMC"] - ac_price_pivot["AC-raw-RQMC"]
ac_price_agreement = pd.DataFrame([{
    "comparison": "AC-Smooth-RQMC minus AC-raw-RQMC",
    "paired_mean_difference": ac_price_difference.mean(),
    "paired_difference_se": ac_price_difference.std(ddof=1) / math.sqrt(len(ac_price_difference)),
    "tolerance": max(0.35, 3.0 * ac_price_difference.std(ddof=1) / math.sqrt(len(ac_price_difference))),
    "pass": abs(ac_price_difference.mean()) <= max(0.35, 3.0 * ac_price_difference.std(ddof=1) / math.sqrt(len(ac_price_difference))),
}])
display(ac_price_summary)
display(ac_price_agreement)
display(ac_greek_summary[ac_greek_summary["active_asset"]])

### 5. Gate decision

In [ ]:
unit_test = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "tests"), "-q"],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)
required_component_rows = (
    2 * 2 * REPLICATIONS * (3 * len(SPOT_BUMP_FRACTIONS) * 2 + 3 * len(VOL_BUMPS) + len(SPOT_BUMP_FRACTIONS) + len(VOL_BUMPS))
)
finite_greeks = bool(np.isfinite(greek_replications["estimate_reported"]).all())
crn_groups = raw_valuations.groupby(["scenario", "method", "replication", "greek", "asset", "bump"])["pricing_seed"].nunique()
crn_pass = bool((crn_groups <= 1).all())
coverage_pass = bool(
    len(greek_replications) == required_component_rows
    and set(greek_replications[greek_replications.greek == "Delta"].asset.unique()) >= {"asset_1", "asset_2", "asset_3"}
    and set(greek_replications[greek_replications.greek == "Vega"].asset.unique()) >= {"asset_1", "asset_2", "asset_3"}
    and set(greek_replications[greek_replications.greek == "Gamma"].asset.unique()) >= {"asset_1", "asset_2", "asset_3"}
    and {"Parallel Vega", "Parallel Gamma"}.issubset(set(greek_replications.greek.unique()))
)
active_gamma_pass = bool(
    len(greek_replications[(greek_replications.greek == "Gamma") & greek_replications.active_asset])
    == 2 * 2 * REPLICATIONS * len(SPOT_BUMP_FRACTIONS)
)
ac_variance_ratio = (
    ac_price_summary.set_index("estimator").loc["AC-raw-RQMC", "value_sd"]
    / ac_price_summary.set_index("estimator").loc["AC-Smooth-RQMC", "value_sd"]
)
rotation_pass = bool(
    ac_price_summary.orthogonality_error_max.max() <= 1e-12
    and ac_price_summary.correlation_error_max.max() <= 1e-12
    and ac_price_summary.minimum_final_loading.min() > 0
    and ac_price_summary.clipping_rate_max.max() <= 0.01
)
gamma_plateau_rate = bump_plateau[bump_plateau.greek.isin(["Gamma", "Parallel Gamma"])].plateau_pass.mean()

gate_summary = pd.DataFrame([
    {"gate": "Day 7 unit tests", "metric": f"returncode={unit_test.returncode}", "threshold": "0", "pass": unit_test.returncode == 0},
    {"gate": "required Delta/Vega/Gamma bump-grid coverage", "metric": f"rows={len(greek_replications)}; expected={required_component_rows}", "threshold": "exact coverage", "pass": coverage_pass},
    {"gate": "CRN / identical scramble within bumped repricing", "metric": f"max_unique_seed={crn_groups.max()}", "threshold": "1", "pass": crn_pass},
    {"gate": "finite estimates and replication uncertainty", "metric": f"finite={finite_greeks}; replications={REPLICATIONS}", "threshold": "all finite; R>=4", "pass": finite_greeks and REPLICATIONS >= 4},
    {"gate": "near-KI and near-autocall separated", "metric": ",".join(sorted(greek_replications.scenario.unique())), "threshold": "both present", "pass": set(greek_replications.scenario.unique()) == {"near-KI", "near-autocall"}},
    {"gate": "active/worst asset and parallel Gamma retained", "metric": f"active_gamma_rows={len(greek_replications[(greek_replications.greek == 'Gamma') & greek_replications.active_asset])}; gamma_plateau_rate={gamma_plateau_rate:.3f}", "threshold": "full rows retained; no plateau may be hidden", "pass": active_gamma_pass},
    {"gate": "bridge-bank seed sensitivity quantified", "metric": f"rows={len(bridge_seed_sensitivity)}; variants={bridge_seed_sensitivity.bank_variant.nunique()}", "threshold": "Delta/Gamma; >=3 variants", "pass": len(bridge_seed_sensitivity) == 18 and bridge_seed_sensitivity.bank_variant.nunique() == 3},
    {"gate": "AC-Smooth price agreement", "metric": f"paired_diff={ac_price_agreement.paired_mean_difference.iloc[0]:.6f}", "threshold": "max(0.35, 3 paired SE)", "pass": bool(ac_price_agreement['pass'].all())},
    {"gate": "AC-Smooth rotation and variance diagnostic", "metric": f"price_SD_ratio_raw/smooth={ac_variance_ratio:.3f}", "threshold": "rotation valid; clipping<=1%; ratio>1", "pass": rotation_pass and ac_variance_ratio > 1.0},
])
DAY7_PASS = bool(gate_summary["pass"].all())
display(gate_summary)
print("DAY 7 STATUS:", "PASS WITH DOCUMENTED GAMMA CAVEATS" if DAY7_PASS else "FAIL")
print("Gamma plateau rate (retained, not used to hide estimates):", gamma_plateau_rate)

### 6. Evidence charts

In [ ]:
palette = {"M0": "#2F6B9A", "M3": "#D97706"}
styles = {"M0": ("o", "-"), "M3": ("s", "--")}
for greek in ("Delta", "Vega", "Gamma"):
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8), constrained_layout=True)
    for ax, scenario_name in zip(axes, ("near-KI", "near-autocall")):
        active_asset = f"asset_{scenarios[scenario_name]['active_asset'] + 1}"
        for method in ("M0", "M3"):
            subset = greek_summary[
                (greek_summary.scenario == scenario_name)
                & (greek_summary.method == method)
                & (greek_summary.greek == greek)
                & (greek_summary.asset == active_asset)
            ].sort_values("bump")
            marker, linestyle = styles[method]
            ax.errorbar(
                subset.bump * 100, subset.estimate_mean, yerr=1.96 * subset.estimate_se,
                marker=marker, linestyle=linestyle, color=palette[method], markerfacecolor="white",
                markeredgewidth=1.5, capsize=3.5, linewidth=1.7, markersize=6.5, label=method,
            )
        ax.axhline(0.0, color="#475569", linewidth=0.9)
        ax.grid(axis="y")
        ax.set_title(scenario_name, loc="left", pad=12)
        ax.set_xlabel("Bump size (volatility points)" if greek == "Vega" else "Relative spot bump (%)")
        ax.set_ylabel("PV per 1% move" if greek != "Gamma" else "PV per squared 1% move")
        ax.legend(ncol=2)
        ax.text(0.0, 1.015, "Bars: 95% replication CI; equal outer N=512, R=4", transform=ax.transAxes, fontsize=8.5, color="#64748b")
    fig.suptitle(f"Active {greek} stability across trigger regimes", x=0.01, ha="left", fontsize=14, fontweight="semibold")
    fig.savefig(OUTPUT_DIR / f"active_{greek.lower()}_bump_stability.png", bbox_inches="tight", facecolor="white")
    plt.show()

heatmap = plateau_rate.pivot_table(index=["scenario", "greek"], columns="method", values="plateau_rate").reindex(columns=["M0", "M3"])
fig, ax = plt.subplots(figsize=(7.6, 6.3), constrained_layout=True)
image = ax.imshow(heatmap.to_numpy(), vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(np.arange(len(heatmap.columns)), heatmap.columns)
ax.set_yticks(np.arange(len(heatmap.index)), [f"{a} | {b}" for a, b in heatmap.index])
for i in range(len(heatmap.index)):
    for j in range(len(heatmap.columns)):
        value = heatmap.iloc[i, j]
        ax.text(j, i, f"{value:.0%}", ha="center", va="center", color="white" if value > 0.55 else "#1f2937", fontweight="semibold")
ax.set_title("Bump-size plateau pass rate", loc="left", pad=28, fontsize=14)
ax.text(0.0, 1.015, "Pure bump-shape criterion; uncertainty-consistency is audited separately", transform=ax.transAxes, fontsize=8.5, color="#64748b")
fig.colorbar(image, ax=ax, label="Plateau pass rate", shrink=0.88)
fig.savefig(OUTPUT_DIR / "greek_plateau_heatmap.png", bbox_inches="tight", facecolor="white")
plt.show()

active_bump = 0.005
ac_active = ac_greek_summary[(ac_greek_summary.active_asset) & (ac_greek_summary.bump == active_bump)]
sd_rows = []
for _, row in ac_price_summary.iterrows():
    sd_rows.append({"estimator": row.estimator, "metric": "Price", "replication_sd": row.value_sd})
for greek in ("Delta", "Gamma"):
    for _, row in ac_active[ac_active.greek == greek].iterrows():
        sd_rows.append({"estimator": row.estimator, "metric": greek, "replication_sd": row.estimate_sd})
ac_sd_comparison = pd.DataFrame(sd_rows)
ac_sd_pivot = ac_sd_comparison.pivot(index="metric", columns="estimator", values="replication_sd").loc[["Price", "Delta", "Gamma"]]
ac_reduction = ac_sd_pivot["AC-raw-RQMC"] / ac_sd_pivot["AC-Smooth-RQMC"]
ac_value_gap = abs(float(ac_price_agreement["paired_mean_difference"].iloc[0]))

fig, ax = plt.subplots(figsize=(8.2, 4.6), constrained_layout=True)
bars = ax.barh(ac_reduction.index, ac_reduction.values, color=["#2F6B9A", "#2A9D8F", "#6B7F3A"], height=0.58)
ax.invert_yaxis()
ax.set_xlabel("Replication-SD reduction factor: raw RQMC / AC-Smooth RQMC")
ax.set_title("AC-Smooth variance reduction", loc="left", pad=28, fontsize=14)
ax.text(0.0, 1.015, f"Isolated next-observation redemption; value gap vs raw RQMC = {ac_value_gap:.6f} per 100", transform=ax.transAxes, fontsize=8.8, color="#64748b")
ax.grid(axis="x")
ax.bar_label(bars, labels=[f"{value:.1f}x" for value in ac_reduction.values], padding=5, fontsize=10, fontweight="semibold")
ax.set_xlim(0, max(ac_reduction.values) * 1.18)
fig.savefig(OUTPUT_DIR / "ac_smooth_sd_reduction.png", bbox_inches="tight", facecolor="white")
plt.show()


### 7. Presentation summary: regime-dependent estimator precision

This matrix condenses the main full-product comparison into one display. It uses
the active asset and the common middle bump: **0.50% relative spot** for Delta
and Gamma, and **0.50 volatility point** for Vega. A ratio above one means M3
has lower replication SD; a ratio below one means M0 has lower replication SD.
The comparison uses the equal outer budget of 512 paths and four replications.

In [ ]:
from matplotlib.colors import TwoSlopeNorm

middle_bump_by_greek = {"Delta": 0.005, "Vega": 0.005, "Gamma": 0.005}
sd_ratio_rows = []
for scenario_name in ("near-KI", "near-autocall"):
    for greek in ("Delta", "Vega", "Gamma"):
        subset = greek_summary[
            (greek_summary.scenario == scenario_name)
            & (greek_summary.greek == greek)
            & (greek_summary.active_asset)
            & (greek_summary.bump == middle_bump_by_greek[greek])
        ].set_index("method")
        m0_sd = float(subset.loc["M0", "estimate_sd"])
        m3_sd = float(subset.loc["M3", "estimate_sd"])
        sd_ratio_rows.append({
            "scenario": scenario_name,
            "greek": greek,
            "bump": middle_bump_by_greek[greek],
            "m0_replication_sd": m0_sd,
            "m3_replication_sd": m3_sd,
            "m0_over_m3_sd_ratio": m0_sd / m3_sd,
            "lower_sd_method": "M3" if m0_sd / m3_sd > 1.0 else "M0",
        })

active_sd_ratio_summary = pd.DataFrame(sd_ratio_rows)
ratio_matrix = (
    active_sd_ratio_summary
    .pivot(index="scenario", columns="greek", values="m0_over_m3_sd_ratio")
    .reindex(index=["near-KI", "near-autocall"], columns=["Delta", "Vega", "Gamma"])
)
log2_ratio = np.log2(ratio_matrix.to_numpy(dtype=float))
colour_limit = max(2.0, float(np.nanmax(np.abs(log2_ratio))))

fig, ax = plt.subplots(figsize=(8.4, 4.1), constrained_layout=True)
image = ax.imshow(log2_ratio, cmap="RdBu", norm=TwoSlopeNorm(vmin=-colour_limit, vcenter=0.0, vmax=colour_limit), aspect="auto")
ax.set_xticks(np.arange(len(ratio_matrix.columns)), ratio_matrix.columns)
ax.set_yticks(np.arange(len(ratio_matrix.index)), ratio_matrix.index)
for row_index, scenario_name in enumerate(ratio_matrix.index):
    for column_index, greek in enumerate(ratio_matrix.columns):
        ratio = float(ratio_matrix.loc[scenario_name, greek])
        winner = "M3 lower SD" if ratio > 1.0 else "M0 lower SD"
        text_colour = "white" if abs(np.log2(ratio)) > 0.65 else "#1f2937"
        ax.text(column_index, row_index, f"{ratio:.2f}x\n{winner}", ha="center", va="center", color=text_colour, fontsize=10, fontweight="semibold")
ax.set_title("M0/M3 active-Greek precision by regime", loc="left", pad=28, fontsize=14)
ax.text(0.0, 1.015, ">1: M3 is more precise; <1: M0 is more precise | middle bump, equal outer N=512, R=4", transform=ax.transAxes, fontsize=8.8, color="#64748b")
colour_bar = fig.colorbar(image, ax=ax, shrink=0.86)
colour_bar.set_label("log2(M0 SD / M3 SD)")
fig.savefig(OUTPUT_DIR / "m0_m3_active_sd_ratio.png", bbox_inches="tight", facecolor="white")
plt.show()

display(active_sd_ratio_summary)


## Takeaways

In [ ]:
method_active = active_summary.pivot_table(
    index=["scenario", "greek", "asset", "bump"], columns="method", values=["estimate_mean", "estimate_se"]
)
print("All requested component and parallel Greeks are saved; see greek_summary.csv.")
print("AC-Smooth raw-RQMC/smoothed price replication-SD ratio:", ac_variance_ratio)
print("Gamma plateau rate:", gamma_plateau_rate)
print("Bridge-seed dispersion:")
display(bridge_seed_dispersion)
print("Interpretation: conditioning and RQMC can reduce replication noise, but near-trigger Gamma remains bump-sensitive. No non-plateau Gamma is removed.")

## Export audit evidence

In [ ]:
exports = {
    "scenario_definitions.csv": scenario_definitions,
    "raw_valuations.csv": raw_valuations,
    "greek_replications.csv": greek_replications,
    "greek_summary.csv": greek_summary,
    "active_asset_greeks.csv": active_summary,
    "active_sd_ratio_summary.csv": active_sd_ratio_summary,
    "bump_plateau.csv": bump_plateau,
    "plateau_rate.csv": plateau_rate,
    "bridge_seed_sensitivity.csv": bridge_seed_sensitivity,
    "bridge_seed_sensitivity_summary.csv": bridge_seed_sensitivity_summary,
    "bridge_seed_dispersion.csv": bridge_seed_dispersion,
    "ac_smooth_valuations.csv": ac_smooth_valuations,
    "ac_smooth_greeks.csv": ac_smooth_greeks,
    "ac_price_summary.csv": ac_price_summary,
    "ac_greek_summary.csv": ac_greek_summary,
    "ac_price_agreement.csv": ac_price_agreement,
    "ac_sd_comparison.csv": ac_sd_comparison,
    "gate_summary.csv": gate_summary,
}
for filename, frame in exports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)
manifest = pd.DataFrame([
    {"field": "status", "value": "PASS WITH DOCUMENTED GAMMA CAVEATS" if DAY7_PASS else "FAIL"},
    {"field": "m0_paths", "value": M0_PATHS},
    {"field": "m0_steps_per_year", "value": M0_STEPS_PER_YEAR},
    {"field": "m3_paths", "value": M3_PATHS},
    {"field": "m3_inner_paths", "value": M3_INNER_PATHS},
    {"field": "m3_bridge_substeps", "value": M3_BRIDGE_SUBSTEPS},
    {"field": "replications", "value": REPLICATIONS},
    {"field": "spot_bumps", "value": ";".join(map(str, SPOT_BUMP_FRACTIONS))},
    {"field": "volatility_bumps", "value": ";".join(map(str, VOL_BUMPS))},
    {"field": "ac_smooth_scope", "value": "next-observation autocall principal redemption only"},
    {"field": "ac_smooth_claim_boundary", "value": "discrete-trigger diagnostic; not Brownian bridge; not full-product replacement"},
    {"field": "gamma_non_plateau_hidden", "value": False},
])
manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)
print("Saved", len(exports) + 1, "CSV files and 6 PNG figures to", OUTPUT_DIR)